# Week 4 · Notebook 1 — The End-to-End ML Workflow
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

AI vs ML vs DL vs GenAI in one picture, then a real supervised-learning pipeline on the classic **Iris** dataset: *data → split → train → evaluate → predict.*

> Runs in Google Colab. No API keys needed.

## 0. Install
scikit-learn and matplotlib are preinstalled in Colab; this is here for a clean local run.

In [ ]:
!pip -q install scikit-learn matplotlib

## 1. The big picture: AI ⊃ ML ⊃ DL ⊃ GenAI
- **AI** — the broad goal of intelligent behavior.
- **ML** — learning patterns from data (this week).
- **DL** — ML with multi-layer neural networks (Week 6).
- **GenAI / LLMs** — DL that *generates* content (back half of the course).

Everything below is a special case of the thing above it.

## 2. Get data
We use **Iris**: 150 flowers, 4 measurements each, 3 species. A tidy table is the starting point of every supervised ML project.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris(as_frame=True)
df = iris.frame                       # features + 'target' column
print('shape:', df.shape)
print('features:', iris.feature_names)
print('classes :', list(iris.target_names))
df.head()

## 3. Explore
Always look before you model. Class balance and feature ranges matter.

In [ ]:
print(df['target'].value_counts().sort_index())   # 50 of each — balanced
df.describe().round(2)

In [ ]:
import matplotlib.pyplot as plt

# Two features colored by species — already fairly separable
plt.figure(figsize=(6, 4))
plt.scatter(df['petal length (cm)'], df['petal width (cm)'],
            c=df['target'], cmap='viridis', edgecolor='k')
plt.xlabel('petal length (cm)'); plt.ylabel('petal width (cm)')
plt.title('Iris — petal length vs width'); plt.show()

## 4. Split: hold out a test set
The single most important habit in ML: **never evaluate on data you trained on.** `stratify=y` keeps the 3 classes balanced in both splits.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('train:', X_train.shape, ' test:', X_test.shape)

## 5. Train (fit) and 6. Evaluate
A k-Nearest-Neighbors classifier: predict a flower's species from its 3 closest neighbors.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)                 # learn
acc = clf.score(X_test, y_test)           # evaluate on held-out data
print(f'test accuracy: {acc:.3f}')

### The cardinal rule, demonstrated
Accuracy on the *training* set is optimistic — it measures memorization, not generalization.

In [ ]:
print(f'train accuracy: {clf.score(X_train, y_train):.3f}  (optimistic!)')
print(f'test  accuracy: {clf.score(X_test,  y_test):.3f}  (what we report)')

## 7. Predict on new data
Hand the trained model a brand-new flower measurement.

In [ ]:
# Build the new sample as a DataFrame with the SAME column names the model
# was fit on — this avoids sklearn's 'feature names' warning.
new_flower = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]],
                          columns=iris.feature_names)  # sepal/petal length & width
pred = clf.predict(new_flower)[0]
print('predicted species:', iris.target_names[pred])

## 8. Iterate
Try `n_neighbors=1`, `5`, `15`. Does test accuracy change? You just did your first hyperparameter tuning — formalized next week.

**Takeaway:** a working supervised model is ~8 lines. Week 5 unpacks *which* model and *which* metric.

In [ ]:
# Your experiment: loop over a few k values and print test accuracy
for k in [1, 3, 5, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    print(f'k={k:>2}  test acc={m.score(X_test, y_test):.3f}')